# Bootstrap


In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [_cwd, *_cwd.parents]
     if (p / "pyproject.toml").is_file() and (p / "research" / "lib").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        "Cannot find Tyrex_PM repo root. Start Jupyter from repo root or research/notebooks/."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


## 1. Objective
Pre-replay counterfactual grid assembly. **This is NOT M2B.5 replay.**


In [ ]:
from research.lib.loaders import load_day
from research.lib.latency import build_latency_prior, write_latency_prior
from research.lib.markets import build_clean_markets, notebook_01_decisions, ws_seq_gap_audit, write_clean_markets
from research.m2b4.pipeline import run_notebook_02, run_notebook_03, run_notebook_04, run_notebook_05, run_notebook_06
from research.m2b4.exploratory import (
    run_notebook_01_exploratory, run_notebook_02_exploratory, run_notebook_03_exploratory,
    run_notebook_04_exploratory, run_notebook_05_exploratory, run_notebook_06_exploratory,
)
from research.lib.exploratory import write_exploratory_trace
from research.lib.buckets import format_decision_output

OUT = REPO_ROOT / "research/output/m2b4"
day = load_day(REPO_ROOT / "var/parquet/date=2026-07-05", load_books=True)
clean = build_clean_markets(day)
gap = ws_seq_gap_audit(day.tables.get("lifecycle_events"))
nb01 = {"decisions": notebook_01_decisions(day, clean, gap)}
prior_path = write_latency_prior(build_latency_prior(default_roots=[REPO_ROOT / "var/runs"]), OUT / "latency_prior.json")
nb02 = run_notebook_02(day, OUT / "clean_markets.csv", prior_path)
nb03 = run_notebook_03(day, OUT / "clean_markets.csv")
nb04 = run_notebook_04(day, OUT / "clean_markets.csv")
nb05 = run_notebook_05(day, OUT / "clean_markets.csv")
strict = run_notebook_06(nb01, nb02, nb03, nb04, nb05)
exp06 = run_notebook_06_exploratory(
    run_notebook_01_exploratory(day, clean, gap),
    run_notebook_02_exploratory(day, OUT / "clean_markets.csv", prior_path),
    run_notebook_03_exploratory(nb03, day, OUT / "clean_markets.csv"),
    run_notebook_04_exploratory(day, OUT / "clean_markets.csv"),
    run_notebook_05_exploratory(day, OUT / "clean_markets.csv"),
)
write_exploratory_trace("06", exp06, OUT)
print("STRICT grid:", strict["decisions"]["m2b5_parameter_grid"])
print("EXPLORATORY grid:", exp06.get("exploratory_grid"))
